In [1]:
import cv2
import mediapipe as mp

In [38]:
import cv2
import mediapipe as mp

mp_drawing=mp.solutions.drawing_utils
mp_object=mp.solutions.objectron
models=['Shoe','Chair','Cup','Camera']
cap=cv2.VideoCapture(1)

with mp_object.Objectron(static_image_mode=False,max_num_objects=5,min_detection_confidence=0.5,min_tracking_confidence=0.5,model_name='Cup') as objectron:
    while cap.isOpened():
        success,image=cap.read()
        image=cv2.rotate(image,cv2.ROTATE_90_CLOCKWISE)
        if not success:
            print("Ignoring empty camera frame")
            continue

        image=cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
        image.flags.writeable=False
        results=objectron.process(image)
        image.flags.writeable=True
        image=cv2.cvtColor(image,cv2.COLOR_RGB2BGR)

        if results.detected_objects:
            print("OBJECT DETECTED")
            print(len(results.detected_objects))

            for detected_object in results.detected_objects:
                mp_drawing.draw_landmarks(image,detected_object.landmarks_2d,mp_object.BOX_CONNECTIONS)
                mp_drawing.draw_axis(image,detected_object.rotation,detected_object.translation)

        cv2.imshow('Mediapipe Objectron',cv2.flip(image,1))

        if cv2.waitKey(5)&0xFF==ord("q"):
            break

cap.release()
cv2.destroyAllWindows()

OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1
OBJECT DETECTED
1


In [42]:
import cv2
import mediapipe as mp

mp_drawing=mp.solutions.drawing_utils
mp_object=mp.solutions.objectron
models=['Shoe','Chair','Cup','Camera']
cap=cv2.VideoCapture(1)
objectrons={}

for model in models:
    objectrons[model]=mp_object.Objectron(
        static_image_mode=False,
        max_num_objects=5,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
        model_name=model
    )
    
print("Objectron models initialized successfully!")
print(f"Available models: {list(objectrons.keys())}")

while cap.isOpened():
    success,image=cap.read()
    if not success:
        print("Ignoring empty camera frame")
        continue
    image=cv2.rotate(image,cv2.ROTATE_90_CLOCKWISE)
    # image=cv2.resize(image,(960,540))
    image=cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    image.flags.writeable=False

    all_results={}

    for model,objectron in objectrons.items():
       all_results[model]=objectron.process(image)

    image.flags.writeable=True

    for model,results in all_results.items():

      if results.detected_objects:

        for detected_object in results.detected_objects:

            mp_drawing.draw_landmarks(
                image,
                detected_object.landmarks_2d,
                mp_object.BOX_CONNECTIONS
            )

            mp_drawing.draw_axis(
                image,
                detected_object.rotation,
                detected_object.translation
            )

            x=int(detected_object.landmarks_2d.landmark[0].x*image.shape[1])
            y=int(detected_object.landmarks_2d.landmark[0].y*image.shape[0])

            cv2.putText(
                image,
                model,
                (x,y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0,255,0),
                2
            )
    image=cv2.cvtColor(image,cv2.COLOR_RGB2BGR)
    cv2.imshow('Mediapipe Objectron',image)

    if cv2.waitKey(1)&0xFF==ord("q"):
      break
for objectron in objectrons.values():
    objectron.close()

cap.release()
cv2.destroyAllWindows()

Objectron models initialized successfully!
Available models: ['Shoe', 'Chair', 'Cup', 'Camera']


In [8]:
import cv2
import mediapipe as mp

mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils

face_mesh = mp_face_mesh.FaceMesh(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    max_num_faces=1
)

drawing_spec = mp_drawing.DrawingSpec(
    thickness=1,
    circle_radius=1,color=(0,255,0)
)

cap = cv2.VideoCapture(1)

while True:

    ret, image = cap.read()
    image=cv2.rotate(image,cv2.ROTATE_90_COUNTERCLOCKWISE)

    if not ret:
        break

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    results = face_mesh.process(image)

    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    if results.multi_face_landmarks:

        for face_landmarks in results.multi_face_landmarks:

            mp_drawing.draw_landmarks(
                image,
                face_landmarks,
                mp_face_mesh.FACEMESH_TESSELATION,
                drawing_spec,
                drawing_spec
            )

    cv2.imshow("Face Mesh", image)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [12]:
import cv2
import mediapipe as mp

mp_face_detect = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils
mp_face_mesh = mp.solutions.face_mesh
mp_drawing_styles = mp.solutions.drawing_styles

face_detection = mp_face_detect.FaceDetection(min_detection_confidence=0.5)

face_mesh = mp_face_mesh.FaceMesh(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    max_num_faces=1
)

drawing_spec = mp_drawing.DrawingSpec(
    thickness=1,
    circle_radius=1,
    color=(0,255,0)
)

cap = cv2.VideoCapture(1)

while True:
    success, frame = cap.read()
    frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
    
    if not success:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    detection_results = face_detection.process(rgb)
    mesh_results = face_mesh.process(rgb)

    # Face Detection
    if detection_results.detections:
        for detection in detection_results.detections:
            mp_drawing.draw_detection(frame, detection)

    # Face Mesh
    if mesh_results.multi_face_landmarks:
        for face_landmarks in mesh_results.multi_face_landmarks:

            mp_drawing.draw_landmarks(
                image=frame,
                landmark_list=face_landmarks,
                connections=mp_face_mesh.FACEMESH_TESSELATION,
                landmark_drawing_spec=drawing_spec,
                connection_drawing_spec=mp_drawing_styles
                    .get_default_face_mesh_tesselation_style()
            )

    cv2.imshow("Face Detection + Face Mesh", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()
face_detection.close()
face_mesh.close()

In [14]:
import cv2
import numpy as np
import mediapipe as mp
import pygame
alaram_path=r"D:\proj-python\music\alarm_1.mp3"

mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

face_mesh = mp_face_mesh.FaceMesh(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    max_num_faces=1
)

drawing_spec = mp_drawing.DrawingSpec(
    thickness=1,
    circle_radius=1,
    color=(0,255,0)
)

face_3d = np.array([
    [0.0, 0.0, 0.0],
    [-30.0, -30.0, -30.0],
    [30.0, -30.0, -30.0],
    [-25.0, 30.0, -20.0],
    [25.0, 30.0, -20.0],
    [0.0, 60.0, -10.0]
], dtype=np.float64)

# EMA smoothing
ALPHA = 0.25

smooth_yaw = 0
smooth_pitch = 0
smooth_roll = 0

cap = cv2.VideoCapture(1)

while True:

    success, frame = cap.read()
    frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)

    if not success:
        break

    h, w, _ = frame.shape

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = face_mesh.process(rgb)

    if results.multi_face_landmarks:

        for face_landmarks in results.multi_face_landmarks:

            lm = face_landmarks.landmark

            indexes = [1, 33, 263, 61, 291, 199]

            face_2d = []

            for idx in indexes:

                x = int(lm[idx].x * w)
                y = int(lm[idx].y * h)

                face_2d.append([x, y])

                cv2.circle(
                    frame,
                    (x, y),
                    4,
                    (0,255,0),
                    -1
                )

            face_2d = np.array(face_2d, dtype=np.float64)

            focal_length = w

            camera_matrix = np.array([
                [focal_length, 0, w / 2],
                [0, focal_length, h / 2],
                [0, 0, 1]
            ], dtype=np.float64)

            dist_coeffs = np.zeros((4, 1))

            success_pnp, rotation_vector, translation_vector = cv2.solvePnP(
                face_3d,
                face_2d,
                camera_matrix,
                dist_coeffs,
                flags=cv2.SOLVEPNP_ITERATIVE
            )

            if success_pnp:

                rotation_matrix, _ = cv2.Rodrigues(
                    rotation_vector
                )

                angles, _, _, _, _, _ = cv2.RQDecomp3x3(
                    rotation_matrix
                )

                pitch = angles[0]
                yaw = angles[1]
                roll = angles[2]

                # EMA smoothing
                smooth_yaw = ALPHA * yaw + (1 - ALPHA) * smooth_yaw
                smooth_pitch = ALPHA * pitch + (1 - ALPHA) * smooth_pitch
                smooth_roll = ALPHA * roll + (1 - ALPHA) * smooth_roll

                # Direction
                if smooth_yaw > 20:
                    direction = "Looking Right"
                
                elif smooth_yaw < -10:
                    direction = "Looking Left"
                
                elif smooth_pitch > 35:
                    direction = "Looking Up"
                
                elif smooth_pitch < -15:
                    direction = "Looking Down"
                    if not pygame.mixer.music.get_busy():
                        pygame.mixer.music.play()
                
                else:
                    direction = "Looking Center"
                    pygame.mixer.music.stop()

                cv2.putText(
                    frame,
                    direction,
                    (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0,255,0),
                    2
                )

                cv2.putText(
                    frame,
                    f"Yaw: {smooth_yaw:.1f}",
                    (30, 90),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (255,255,255),
                    2
                )

                cv2.putText(
                    frame,
                    f"Pitch: {smooth_pitch:.1f}",
                    (30,120),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (255,255,255),
                    2
                )

                cv2.putText(
                    frame,
                    f"Roll: {smooth_roll:.1f}",
                    (30,150),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (255,255,255),
                    2
                )

            mp_drawing.draw_landmarks(
                frame,
                face_landmarks,
                mp_face_mesh.FACEMESH_TESSELATION,
                drawing_spec,
                mp_drawing_styles.get_default_face_mesh_tesselation_style()
            )

    cv2.imshow("Driver Head Pose", frame)

    if cv2.waitKey(1) & 0xFF == 27 or cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
face_mesh.close()